In *LangChain*, a **tool** is a specialized function or interface that allows an LLM to interact with the outside world, effectively acting as an extension of its capabilities. By binding tools to an LLM, you enable it to perform tasks it couldn't do based on training data alone, such as fetching real-time weather data , performing internet searches, or executing custom logic.

### **Key Concepts for Tools:**
* **Functionality:** Tools are typically defined as standard Python functions but marked with the `@tool` decorator, which converts them into a format that the LLM can understand and trigger.
* **The Importance of Docstrings:** Providing a descriptive **docstring** for your function is critical. The LLM relies on this description to understand exactly what the tool does, what arguments it requires, and when it is appropriate to use it.
* **Binding to LLMs:** Once defined, the tool is bound to the LLM. When a user provides input, the LLM analyzes it and, if necessary, decides to make a **tool call** instead of generating a direct text response.
* **Tool Execution:** When a tool call is triggered, the result is captured as a **Tool Message**. This output is then fed back to the LLM as context, allowing it to synthesize the final answer for the user.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000239E1F08160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000239E1F08070>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """ Get the weather at the location """
    return f"It's a sunny in {location}"

#One way to bind tool with model
model_with_tool = model.bind_tools([get_weather])

response = model_with_tool.invoke("Whats the weather in boston?")

print(response)

for tool in response.tool_calls:
    print(f"Tool : {tool['name']}")
    print(f"Args : {tool['args']}")
#another way to bind tool with model
# from langchain.agents import create_agent

# agent = create_agent(
#     model="google_genai:gemini-2.5-flash-lite",
#     tools=[get_weather],
#     system_prompt="You are a helpful assistant",
# )

content='' additional_kwargs={'tool_calls': [{'id': '6whq9mcm7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.024205475, 'completion_tokens_details': None, 'prompt_time': 0.014358073, 'prompt_tokens_details': None, 'queue_time': 0.047085456, 'total_time': 0.038563548}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f0917-9c36-7da0-ac9d-631b8ba81e2d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '6whq9mcm7', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}
Tool : get_weather
Args : {'location': 'Boston'}
